# Monitoring controller

> Coordinates monitoring sessions, worker threads, scheduled lifecycle events, startup housekeeping, and report access.

`MonitorController` owns the runtime state for process and foreground monitoring. It starts and stops database sessions, manages the tracker threads, performs startup cleanup, and exposes report data for the current or most recently completed session.

Lifecycle operations are serialized with a re-entrant lock. Midnight rollover and timed pause/resume operations close the current session and start a new one, while optional callbacks allow the tray or UI layer to receive notifications and state changes.

In [ ]:
#| default_exp monitoring_controller

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| exporti
import logging
logger = logging.getLogger(__name__)


In [ ]:
#| export
from datetime import datetime, timedelta
import threading
import uuid
from fastcore.basics import patch

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import snooper_pkg.config as cf
from snooper_pkg.db import *
from snooper_pkg.reporter import *
from snooper_pkg.foreground_tracker import *
from snooper_pkg.process_monitor import *

ModuleNotFoundError: No module named 'snooper_pkg.reporter'

## Scheduling helpers

In [ ]:
#| export
def seconds_until_midnight(now=None):
    "Return the number of seconds from `now` until the next midnight."
    now = now or datetime.now()
    tomorrow = (now + timedelta(days=1)).date()
    midnight = datetime.combine(tomorrow, datetime.min.time())
    return (midnight - now).total_seconds()

## Controller state

In [ ]:
#| export
class MonitorController:
    "Own monitoring session state, worker threads, timers, and UI callbacks."

    def __init__(self, debug = True):
        "Initialize controller state and run startup housekeeping."
        self.running = False
        self.session_start = None
        self.session_stop = None
        self.stop_event = None
        self.monitor_threads = {}
        self.session_id = None
        self.debug = debug
        self.pause_until = None
        self.resume_timer = None
        # place holder call backs from tray
        self.on_notify = None
        self.on_state_change = None
        self.rollover_timer = None
        self.lifecycle_lock = threading.RLock()
        self.run_startup_housekeeping()

## Session lifecycle

In [ ]:
#| export
@patch
def _midnight_rollover(
    self: MonitorController,  # Controller handling the scheduled rollover
):
    "Close the active session at midnight and immediately start another."
    with self.lifecycle_lock:
        if self.running:
            self.stop_monitoring(stop_reason = "midnight_rollover")
            self.start_monitoring()
            if self.on_notify: self.on_notify("Monitoring is renewed on day break")
            if self.debug: logger.debug("Monitoring auto-resumed after day break")
            if self.on_state_change: self.on_state_change()

In [ ]:
#| export
@patch
def start_monitoring(
    self: MonitorController,  # Controller whose monitoring should start
):
    "Start a new session and its process and foreground monitoring threads."
    with self.lifecycle_lock:
        if self.running:
            if self.debug:logger.debug("Monitoring already running")
            return

        self.running = True
        self.session_id = str(uuid.uuid4())
        self.session_start = datetime.now()
        start_session(self.session_id, self.session_start)

        self.stop_event = threading.Event()

        self.monitor_threads['process'] = threading.Thread(
            target=start_process_monitoring,
            args=(self.session_id, self.stop_event, cf.EXCLUDED_PROCESSES),
            daemon=True,
        )
        self.monitor_threads['foreground'] = threading.Thread(
            target=start_foreground_app_monitoring, 
            args=(self.session_id, self.stop_event), 
            daemon=True
        )

        for mt in self.monitor_threads.values(): mt.start()
        # start counter till until midnight    
        self.rollover_timer = threading.Timer(seconds_until_midnight(), self._midnight_rollover)
        self.rollover_timer.daemon = True
        self.rollover_timer.start()

        if self.debug: logger.debug(f"Session started at {self.session_start}")

In [ ]:
#| export
@patch
def stop_monitoring(
    self: MonitorController,  # Controller whose active session should stop
    stop_reason = "None",
):
    "Stop active workers, close the session, and record its stop reason."
    with self.lifecycle_lock:
        if not self.running:
            if self.debug:logger.debug("Monitoring is not running")
            return

        self.running = False
        self.session_stop = datetime.now()
        
        self.stop_event.set()
        try:
            for k, mt in self.monitor_threads.items():
                mt.join(timeout=5)
                if self.debug: logger.debug(f"Stopped monitoring for: {k}")
        finally:
            end_session(self.session_id, self.session_stop, stop_reason)
            self.monitor_threads = {}
            self.session_id = None
            if self.rollover_timer:
                self.rollover_timer.cancel()
                self.rollover_timer = None

        if self.debug:logger.debug(f"Session stopped at {self.session_stop}")
        if self.debug:logger.debug(f"Duration: {self.session_stop - self.session_start}")

In [ ]:
#| export
@patch
def is_running(
    self: MonitorController,  # Controller whose running state is requested
):
    "Return whether a monitoring session is currently running."
    return self.running

## Startup housekeeping

In [ ]:
#| export
@patch
def run_startup_housekeeping(
    self: MonitorController,  # Controller performing startup cleanup
):
    "Close stale sessions and delete data older than the retention cutoff."
    now = datetime.now()
    closed_sessions = close_stale_open_sessions(now)
    cutoff_ts = delete_old_data(now)

    if self.debug:
        logger.debug(f"Closed stale sessions: {closed_sessions}")
        logger.debug(f"Deleted data older than: {cutoff_ts}")

## Report data

In [ ]:
#| export
@patch
def generate_last_report_data(
    self: MonitorController,  # Controller requesting the latest completed report
):
    "Build report data for the latest completed session, or return `{}`."
    session = get_last_completed_session()
    if session:
        events = get_foreground_events(session["session_id"])
        return build_session_report_data(session, events)
    else: return {}

@patch
def generate_current_session_stats(
    self: MonitorController,  # Controller requesting its active session report
):
    "Build report data for the active session, or return `{}` when none exists."
    if self.session_id is not None:
        session = get_session_by_session_id(self.session_id)
        events = get_foreground_events(self.session_id)
        return build_session_report_data(session, events)
    else: return {}

## Pause and resume

In [ ]:
#| export
@patch
def pause_for(
    self: MonitorController,  # Controller whose monitoring should be paused
    minutes=None,
):
    "Stop active monitoring, schedule automatic resume, and return the pause end time."
    with self.lifecycle_lock:
        if minutes is None: minutes = getattr(cf, "DEFAULT_PAUSE_MINUTES", 30)
        if self.resume_timer: self.resume_timer.cancel()
        if self.running: self.stop_monitoring(stop_reason="manual_pause")

        self.pause_until = datetime.now() + timedelta(minutes=minutes)

        self.resume_timer = threading.Timer(minutes * 60, self._auto_resume)
        self.resume_timer.daemon = True
        self.resume_timer.start()

        if self.debug:
            logger.debug(f"Monitoring paused until {self.pause_until}")

        return self.pause_until

@patch
def is_paused(
    self: MonitorController,  # Controller whose pause state is requested
):
    "Return whether the configured pause end time is still in the future."
    if self.pause_until is None: return False
    return datetime.now() < self.pause_until

In [ ]:
#| export
@patch
def _auto_resume(
    self: MonitorController,  # Controller completing a scheduled resume
):
    "Clear pause state, start monitoring if stopped, and emit resume callbacks."
    with self.lifecycle_lock:
        self.resume_timer = None
        self.pause_until = None

        if not self.running: self.start_monitoring()
        if self.on_notify: self.on_notify("Monitoring is starting again")
        if self.debug: logger.debug("Monitoring auto-resumed")
        if self.on_state_change: self.on_state_change()

@patch
def resume_now(
    self: MonitorController,  # Controller whose monitoring should resume
):
    "Cancel any pending resume timer and resume monitoring immediately."
    with self.lifecycle_lock:
        if self.resume_timer: self.resume_timer.cancel()
        self._auto_resume()

- `stop_monitoring()` uses the literal string `"None"` as its default
  `stop_reason`. Confirm whether this should remain `"None"`, become Python
  `None`, or use a defined reason such as `"unknown"`.

- Constructing `MonitorController` immediately runs database housekeeping.
  Confirm that this constructor side effect is intentional.

- `resume_now()` calls `_auto_resume()` even when no pause is active. If the
  controller was manually stopped, this starts a new monitoring session and
  emits resume callbacks. Confirm that this is the intended distinction
  between “resume” and “start”.

- The callback contracts are implicit: `on_notify` is called with one string,
  while `on_state_change` is called without arguments. They may also be invoked
  from timer threads. Confirm that the tray layer safely transfers any Tkinter
  work to the Tk main thread.

- `start_monitoring()` does not reset `session_stop`, so while a later session
  is running that attribute may still contain the previous session's stop
  time. Confirm whether consumers rely on it being `None` during an active
  session.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()